In [1]:
!pip install -q uv fastapi uvicorn
!uv venv --python 3.11 /kaggle/working/bfclenv
!uv pip install --python /kaggle/working/bfclenv/bin/python bfcl-eval
!/kaggle/working/bfclenv/bin/python -c "import bfcl_eval, sys; print('python', sys.version.split()[0], '| bfcl_eval at', bfcl_eval.__path__[0])"

Using CPython 3.11.18
Creating virtual environment at: bfclenv
Activate with: source bfclenv/bin/activate
Using Python 3.11.18 environment at: bfclenv
Resolved 150 packages in 1.78s                                       
Prepared 149 packages in 35.59s                                          
░░░░░░░░░░░░░░░░░░░░ [0/150] Installing wheels...                               warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 150 packages in 13.79s                            
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.6
 + annotated-types==0.8.0
 + anthropic==2.10.0
 + anyio==4.16.0
 + argcomplete==3.7.2
 + attrs==26.1.0
 + beautifulsoup4==4.15.0
 + bfcl-eval==2026.

In [2]:
  !uv pip install --python /kaggle/working/bfclenv/bin/python soundfile
  !/kaggle/working/bfclenv/bin/bfcl --help | head -5

Using Python 3.11.18 environment at: bfclenv
Resolved 5 packages in 791ms                                         
Prepared 1 package in 30ms                                               
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 41ms                                 
 + soundfile==0.14.0
                                                                                
 Usage: bfcl [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮

In [3]:
import os, pathlib, shutil, subprocess

ROOT = pathlib.Path("/kaggle/working/bfcl_root"); ROOT.mkdir(parents=True, exist_ok=True)
VENV_PY = "/kaggle/working/bfclenv/bin/python"
BFCL    = "/kaggle/working/bfclenv/bin/bfcl"
PORT    = 1053

pkg_dir = subprocess.check_output(
    [VENV_PY, "-c", "import bfcl_eval; print(bfcl_eval.__path__[0])"], text=True).strip()
shutil.copy(f"{pkg_dir}/.env.example", ROOT / ".env")

BFCL_ENV = {**os.environ,
            "BFCL_PROJECT_ROOT": str(ROOT),
            "LOCAL_SERVER_ENDPOINT": "localhost",
            "LOCAL_SERVER_PORT": str(PORT)}

def run_bfcl(args):
    """Run the bfcl CLI in its own env and stream its output into this cell."""
    p = subprocess.Popen(args, env=BFCL_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print(f"\n[exit code {p.returncode}]")
    return p.returncode

run_bfcl([BFCL, "--help"])

                                                                                
 Usage: bfcl [OPTIONS] COMMAND [ARGS]...                                        
                                                                                
╭─ Options ────────────────────────────────────────────────────────────────────╮
│ --install-completion          <bool>  Install completion for the current     │
│                                       shell.                                 │
│ --show-completion             <bool>  Show completion for the current shell, │
│                                       to copy it or customize the            │
│                                       installation.                          │
│ --help                -h      <bool>  Show this message and exit.            │
╰──────────────────────────────────────────────────────────────────────────────╯
╭─ Commands ───────────────────────────────────────────────────────────────────╮
│ models           List avai

0

In [4]:
import time, uuid
from fastapi import FastAPI, Request
from starlette.concurrency import run_in_threadpool

def make_app(generate_fn, model_id):
    """generate_fn(prompt, temperature, max_tokens) -> (text, n_prompt_tokens, n_completion_tokens, finish_reason)"""
    app = FastAPI()

    @app.get("/v1/models")
    def list_models():
        return {"object": "list", "data": [{"id": model_id, "object": "model", "owned_by": "local"}]}

    @app.post("/v1/completions")
    async def completions(request: Request):
        body = await request.json()
        prompt = body["prompt"]
        if isinstance(prompt, list):
            prompt = prompt[0]
        text, n_in, n_out, finish = await run_in_threadpool(
            generate_fn, prompt, body.get("temperature", 0.0), body.get("max_tokens", 256))
        return {
            "id": "cmpl-" + uuid.uuid4().hex[:12],
            "object": "text_completion",
            "created": int(time.time()),
            "model": body.get("model", model_id),
            "choices": [{"text": text, "index": 0, "logprobs": None, "finish_reason": finish}],
            "usage": {"prompt_tokens": n_in, "completion_tokens": n_out, "total_tokens": n_in + n_out},
        }

    return app

In [5]:
import gc, threading, torch, time
from transformers import AutoTokenizer, AutoModelForCausalLM
import uvicorn

MODEL_ID        = "Qwen/Qwen3-1.7B"
WEIGHTS         = MODEL_ID
NO_THINK_PREFIX = True
MAX_NEW_TOKENS_CAP = 768

if "server" in globals():
    server.should_exit = True; time.sleep(3)
for _n in ("model", "tok"):
    if _n in globals():
        del globals()[_n]
gc.collect(); torch.cuda.empty_cache()

tok = AutoTokenizer.from_pretrained(WEIGHTS)
model = AutoModelForCausalLM.from_pretrained(
    WEIGHTS, torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa").eval()
print(f"model on GPU 0: {torch.cuda.memory_allocated(0)/1e9:.1f} GB")

GEN_LOCK = threading.Lock()
THINK_OFF = "\n\n\n\n"
ASSISTANT_TAG = "<|im_start|>assistant\n"
CAREFUL = None

def generate_text(prompt, temperature, max_tokens):
    if CAREFUL and prompt.startswith("<|im_start|>system\n"):
        prompt = prompt.replace("<|im_start|>system\n",
                                "<|im_start|>system\n" + CAREFUL + "\n\n", 1)
    if NO_THINK_PREFIX and prompt.endswith(ASSISTANT_TAG):
        prompt = prompt + THINK_OFF
    enc = tok(prompt, return_tensors="pt").to(model.device)
    n_in = enc["input_ids"].shape[1]
    kw = dict(max_new_tokens=min(int(max_tokens), MAX_NEW_TOKENS_CAP), pad_token_id=tok.pad_token_id)
    if temperature and temperature > 0.01:
        kw.update(do_sample=True, temperature=float(temperature))
    else:
        kw.update(do_sample=False)
    with GEN_LOCK, torch.no_grad():
        out = model.generate(**enc, **kw)
    new = out[0][n_in:]
    text = tok.decode(new, skip_special_tokens=True)
    finish = "length" if len(new) >= kw["max_new_tokens"] else "stop"
    return text, n_in, len(new), finish

app = make_app(generate_text, MODEL_ID)
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("server started on port", PORT)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (3 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

model on GPU 0: 3.4 GB
server started on port 1053


In [6]:
CAREFUL = ("You are a tool-using assistant. Only call a tool when the user's request fits one of the provided tools "
           "AND every required argument is given in the conversation. Never invent, guess, or default argument values. "
           "If a required argument is missing, ask the user for it instead of calling the tool. "
           "If no provided tool can do what the user asks, say you can't do that. "
           "Never state a tool's result unless you actually called the tool.")

IRRFAM  = ["irrelevance", "live_irrelevance", "live_relevance"]
RUN_TAG = "base_1p7b_careful_irrfam"
RESULT_DIR = ROOT / f"result_{RUN_TAG}"
SCORE_DIR  = ROOT / f"score_{RUN_TAG}"

rc = run_bfcl([BFCL, "generate", "--model", MODEL_NAME, "--skip-server-setup",
               "--test-category", ",".join(IRRFAM),
               "--result-dir", str(RESULT_DIR), "--num-threads", "4"]); assert rc == 0
rc = run_bfcl([BFCL, "evaluate", "--model", MODEL_NAME,
               "--test-category", ",".join(IRRFAM),
               "--result-dir", str(RESULT_DIR), "--score-dir", str(SCORE_DIR)]); assert rc == 0

CAREFUL = None   # reset so later cells are back to the plain base protocol

NameError: name 'MODEL_NAME' is not defined

In [8]:
import pathlib

CATS = ["simple_python", "multiple", "parallel", "parallel_multiple", "irrelevance",
        "live_simple", "live_multiple", "live_parallel", "live_parallel_multiple",
        "live_irrelevance", "live_relevance"]

dev_ids = ROOT / "test_case_ids_to_generate.json"
if dev_ids.exists():
    dev_ids.rename(ROOT / "dev256_ids.json")
    print("archived dev IDs ->", ROOT / "dev256_ids.json")

total = 0
for cat in CATS:
    n = sum(1 for l in open(pathlib.Path(pkg_dir) / "data" / f"BFCL_v4_{cat}.json") if l.strip())
    total += n
    print(f"{cat:28s} {n}")
print("TOTAL expected:", total)   # 3491

simple_python                400
multiple                     200
parallel                     200
parallel_multiple            200
irrelevance                  240
live_simple                  258
live_multiple                1053
live_parallel                16
live_parallel_multiple       24
live_irrelevance             884
live_relevance               16
TOTAL expected: 3491


In [ ]:
import requests, time

MODEL_NAME = "Qwen/Qwen3-1.7B-FC"
RUN_TAG    = "base_1p7b_nothink_full"
RESULT_DIR = ROOT / f"result_{RUN_TAG}"
SCORE_DIR  = ROOT / f"score_{RUN_TAG}"

assert requests.get(f"http://127.0.0.1:{PORT}/v1/models", timeout=5).status_code == 200

t0 = time.time()
rc = run_bfcl([BFCL, "generate", "--model", MODEL_NAME, "--skip-server-setup",
               "--test-category", ",".join(CATS),
               "--result-dir", str(RESULT_DIR), "--num-threads", "4"])
dt = time.time() - t0
print(f"generation took {dt:.0f}s ({dt/3600:.2f} h) | exit {rc}")
assert rc == 0

Generating results for ['Qwen/Qwen3-1.7B-FC']
Running full test cases for categories: ['irrelevance', 'live_irrelevance', 'live_multiple', 'live_parallel', 'live_parallel_multiple', 'live_relevance', 'live_simple', 'multiple', 'parallel', 'parallel_multiple', 'simple_python'].
Max context length: 40960
server is ready!

Generating results for Qwen/Qwen3-1.7B-FC:  98%|█████████▊| 3431/3491 [3:18:50<02:02,  2.04s/it]


In [ ]:
import pandas as pd
from IPython.display import display

rc = run_bfcl([BFCL, "evaluate", "--model", MODEL_NAME,
               "--test-category", ",".join(CATS),
               "--result-dir", str(RESULT_DIR), "--score-dir", str(SCORE_DIR)])
assert rc == 0

for name in ["data_overall.csv", "data_non_live.csv", "data_live.csv"]:
    p = SCORE_DIR / name
    if p.exists():
        print("\n==", name); display(pd.read_csv(p))
    else:
        print("missing", p)

In [ ]:
import zipfile, json
from datetime import datetime, timezone

manifest = {
    "run_tag": RUN_TAG, "model": MODEL_NAME, "weights": "Qwen/Qwen3-1.7B",
    "bfcl_eval_version": "2026.3.23", "categories": CATS,
    "protocol": {"max_new_tokens_cap": 768, "decoding": "greedy (temp<=0.01)",
                 "no_think_prefix": True, "dtype": "fp16", "attn": "sdpa"},
    "generated_at": datetime.now(timezone.utc).isoformat(),
}
mpath = ROOT / f"manifest_{RUN_TAG}.json"
mpath.write_text(json.dumps(manifest, indent=2))

zf = f"/kaggle/working/bfcl_{RUN_TAG}.zip"
with zipfile.ZipFile(zf, "w", zipfile.ZIP_DEFLATED) as z:
    for d in (RESULT_DIR, SCORE_DIR):
        for p in d.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(ROOT))
    z.write(mpath, mpath.name)
print("saved:", zf, pathlib.Path(zf).stat().st_size / 1e6, "MB")

In [ ]:
CAREFUL = ("You are a tool-using assistant. Only call a tool when the user's request fits one of the provided tools "
           "AND every required argument is given in the conversation. Never invent, guess, or default argument values. "
           "If a required argument is missing, ask the user for it instead of calling the tool. "
           "If no provided tool can do what the user asks, say you can't do that. "
           "Never state a tool's result unless you actually called the tool.")

IRRFAM  = ["irrelevance", "live_irrelevance", "live_relevance"]
RUN_TAG = "base_1p7b_careful_irrfam"
RESULT_DIR = ROOT / f"result_{RUN_TAG}"
SCORE_DIR  = ROOT / f"score_{RUN_TAG}"

rc = run_bfcl([BFCL, "generate", "--model", MODEL_NAME, "--skip-server-setup",
               "--test-category", ",".join(IRRFAM),
               "--result-dir", str(RESULT_DIR), "--num-threads", "4"]); assert rc == 0
rc = run_bfcl([BFCL, "evaluate", "--model", MODEL_NAME,
               "--test-category", ",".join(IRRFAM),
               "--result-dir", str(RESULT_DIR), "--score-dir", str(SCORE_DIR)]); assert rc == 0
CAREFUL = None